In [ ]:
from loadforecasting_models import Gam
from pygam import s, te, f


In [12]:
import pandas as pd

# Read CSV and parse datetime
#
df_load = pd.read_csv(
    '../data/energy_data.csv',
    sep=',',
    parse_dates=['date'],
    dayfirst=False,
).set_index('date')

net_load_profile = df_load['Consumption'] - df_load['Production']
net_load_profile


date
2025-01-01 00:00:00+01:00    126.95957
2025-01-01 00:15:00+01:00    155.28748
2025-01-01 00:30:00+01:00    128.95338
2025-01-01 00:45:00+01:00    127.86432
2025-01-01 01:00:00+01:00    155.90400
                               ...    
2026-02-22 16:00:00+01:00    -72.99824
2026-02-22 16:15:00+01:00    -94.17236
2026-02-22 16:30:00+01:00    -61.19013
2026-02-22 16:45:00+01:00     82.83377
2026-02-22 17:00:00+01:00     83.85065
Length: 40101, dtype: float64

In [ ]:
import importlib
import get_weather_data
importlib.reload(get_weather_data)


weather_data = get_weather_data.WeatherDataRetriever().retrieve_weather_data(
    start_date="2025-01-01",
    end_date="2026-01-01",
    weather_actuality="actual",
    )


In [ ]:

all_gam_terms = (
    s(features.index("lag_7_days")) +
    f(features.index("day_of_week")) +
    te(
        s(features.index("minute_of_day"), n_splines=5),
        s(features.index("temperature_2m"), n_splines=5),
    ) +
    te(
        s(features.index("minute_of_day"), n_splines=10, lam=0.3),
        s(features.index("global_tilted_irradiance"), n_splines=10, lam=0.15),
    ) +
    te(
        s(features.index("temperature_2m"), n_splines=10, lam=0.3),
        s(features.index("wind_speed_10m"), n_splines=10, lam=0.15),
    ))

my_model = Gam(all_gam_terms, normalizer=normalizer)
if do_training:
    history = my_model.train_model(x_train, y_train)
if do_evaluation:
    history = my_model.evaluate(x_test, y_test, results=history, de_normalize=True)
